<a href="https://colab.research.google.com/github/sonu786786/Fundamentals-of-Artificial-Intelligence/blob/main/Lab_08/Jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import math
import random

# Node Class
class MCTSNode:
    def __init__(self, name, parent=None):
        self.name     = name
        self.parent   = parent
        self.children = []
        self.visits   = 0
        self.wins     = 0.0

    def is_fully_expanded(self):
        return len(self.children) > 0

    def is_leaf(self):
        return len(self.children) == 0

    def ucb1(self, exploration=math.sqrt(2)):
        """Upper Confidence Bound 1 formula."""
        if self.visits == 0:
            return float('inf')          # Unvisited → highest priority
        return (self.wins / self.visits) + \
               exploration * math.sqrt(math.log(self.parent.visits) / self.visits)

    def best_child(self):
        """Select child with highest UCB1 score."""
        return max(self.children, key=lambda c: c.ucb1())

    def __repr__(self):
        win_rate = round(self.wins / self.visits, 4) if self.visits > 0 else 0
        return (f"Node({self.name!r:6} | visits={self.visits} | "
                f"wins={self.wins} | win_rate={win_rate})")


# Build Game Tree
def build_tree():
    """
    Tree structure:
         Root
        /    \\
       A      B
      / \\
    A1   A2
    """
    root = MCTSNode("Root")
    A    = MCTSNode("A",  parent=root)
    B    = MCTSNode("B",  parent=root)
    A1   = MCTSNode("A1", parent=A)
    A2   = MCTSNode("A2", parent=A)

    root.children = [A, B]
    A.children    = [A1, A2]
    # B and A1/A2 are leaf nodes (no children)

    return root, {"Root": root, "A": A, "B": B, "A1": A1, "A2": A2}


# MCTS Steps

# STEP 1 — Selection
def selection(node):
    """
    Traverse the tree using UCB1 until a leaf or unexpanded node is reached.
    """
    path = [node]
    print("\n STEP 1 — SELECTION")
    print(f"   Start at: {node.name}")

    while not node.is_leaf():
        # If any child is unvisited → it needs expansion first
        unvisited = [c for c in node.children if c.visits == 0]
        if unvisited:
            print(f"   → '{node.name}' has unvisited children → stop here for expansion")
            break
        node = node.best_child()
        path.append(node)
        print(f"   → Selected '{node.name}' (UCB1 = {node.ucb1():.4f})")

    print(f"   Selected node for expansion: '{node.name}'")
    return node, path


# STEP 2 — Expansion
def expansion(node):
    """
    Expand an unvisited child of the selected node.
    """
    print("\n STEP 2 — EXPANSION")
    unvisited = [c for c in node.children if c.visits == 0]

    if not unvisited:
        print(f"   '{node.name}' is a terminal leaf — no expansion needed.")
        return node

    # Pick the first unvisited child (deterministic for clarity)
    child = unvisited[0]
    print(f"   Expanding child '{child.name}' from '{node.name}'")
    return child


# STEP 3 — Simulation (Rollout)
def simulation(node):
    """
    Random playout from the expanded node → returns a simulated result (0 or 1).
    """
    print("\n STEP 3 — SIMULATION (Random Rollout)")
    result = random.choice([0, 1])        # 0 = loss, 1 = win
    print(f"   Simulating from '{node.name}' → result = {result} "
          f"({'Win ' if result == 1 else 'Loss '})")
    return result


# STEP 4 — Backpropagation
def backpropagation(node, result, path):
    """
    Propagate the simulation result up the path to the root.
    """
    print("\n STEP 4 — BACKPROPAGATION")
    # Build the full backprop path: from expanded node up through selected path
    backprop_path = path + ([node] if node not in path else [])

    for n in reversed(backprop_path):
        n.visits += 1
        n.wins   += result
        print(f"   Updated '{n.name}': visits={n.visits}, wins={n.wins}")


# Print Tree
def print_tree(node, indent=0):
    prefix = "   " * indent + ("└─ " if indent > 0 else "")
    print(f"{prefix}{node}")
    for child in node.children:
        print_tree(child, indent + 1)


# Run One Full MCTS Iteration
def run_mcts(seed=42):
    random.seed(seed)

    print("=" * 60)
    print("   Monte Carlo Tree Search — One Full Iteration")
    print("=" * 60)

    # Build tree
    root, nodes = build_tree()

    print("\n Initial Tree State:")
    print_tree(root)

    # --- MCTS Steps ---
    selected_node, path  = selection(root)
    expanded_node        = expansion(selected_node)
    result               = simulation(expanded_node)
    backpropagation(expanded_node, result, path)

    # --- Final State ---
    print("\n" + "=" * 60)
    print(" Updated Tree State After One MCTS Iteration:")
    print("=" * 60)
    print_tree(root)

    print("\n Node Summary Table:")
    print(f"{'Node':<8} {'Visits':<10} {'Wins':<8} {'Win Rate':<12} {'UCB1'}")
    print("-" * 55)
    for name, node in nodes.items():
        win_rate = round(node.wins / node.visits, 4) if node.visits > 0 else "-"
        ucb1_val = round(node.ucb1(), 4) if node.visits > 0 and node.parent else "-"
        print(f"{name:<8} {node.visits:<10} {node.wins:<8} {str(win_rate):<12} {ucb1_val}")

# Entry Point
run_mcts(seed=42)

   Monte Carlo Tree Search — One Full Iteration

 Initial Tree State:
Node('Root' | visits=0 | wins=0.0 | win_rate=0)
   └─ Node('A'    | visits=0 | wins=0.0 | win_rate=0)
      └─ Node('A1'   | visits=0 | wins=0.0 | win_rate=0)
      └─ Node('A2'   | visits=0 | wins=0.0 | win_rate=0)
   └─ Node('B'    | visits=0 | wins=0.0 | win_rate=0)

 STEP 1 — SELECTION
   Start at: Root
   → 'Root' has unvisited children → stop here for expansion
   Selected node for expansion: 'Root'

 STEP 2 — EXPANSION
   Expanding child 'A' from 'Root'

 STEP 3 — SIMULATION (Random Rollout)
   Simulating from 'A' → result = 0 (Loss )

 STEP 4 — BACKPROPAGATION
   Updated 'A': visits=1, wins=0.0
   Updated 'Root': visits=1, wins=0.0

 Updated Tree State After One MCTS Iteration:
Node('Root' | visits=1 | wins=0.0 | win_rate=0.0)
   └─ Node('A'    | visits=1 | wins=0.0 | win_rate=0.0)
      └─ Node('A1'   | visits=0 | wins=0.0 | win_rate=0)
      └─ Node('A2'   | visits=0 | wins=0.0 | win_rate=0)
   └─ Node('B' 